# 3 - Generate Outcome Data

This notebook prepares gene expression outcome datasets from the LEAP spatial transcriptomics cohort so they can be used later for model training and evaluation (e.g., predicting gene expression from histology embeddings).

It loads GeoMx expression data and sample metadata, processes it, selects gene subsets, and exports cleaned outcome matrices for downstream modelling.

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import pandas as pd
import numpy as np
import os
from LEAPdata import get_slide_id, get_leap_ids

# Load the spatial data

In [ ]:
OME_LEVEL = 2 
DOWNSAMPLE = 2**OME_LEVEL
segment='CD45+'

root_path = "../data"
rawdata_path = os.path.join(root_path,'01-raw')
outcome_path = os.path.join(root_path,'02-input','outcomes')

os.makedirs(outcome_path, exist_ok=True)

In [7]:
rawdata_path = "C:/Users/hooll/Dropbox/19 PostGrad/00 CB Lab/01 projects/06 MultiModal/data/01-raw"

In [8]:

sample_metadata = pd.read_excel(os.path.join(rawdata_path,'metadata',"meta_data.xlsx"), index_col=0)
samples = pd.read_excel(os.path.join(rawdata_path,'metadata',"samples.xlsx"), index_col=0)

In [9]:
samples.head()

,Biobank_ID,Slide_ID,H&E,Response,ECT_Carbo
LEAP_ID,,,,,
LEAP001,19005858,1,YES,Non-Responder,NO
LEAP003,21002266,2,YES,Non-Responder,NO
LEAP005,22001707,3,YES,Non-Responder,NO
LEAP008,22001709,4,YES,Responder,YES
LEAP009,19005859,5,YES,Responder,NO


In [11]:
# Support metadata with DCC_ID either as a column or already as the index.
if 'DCC_ID' in sample_metadata.columns:
    sample_metadata = sample_metadata.set_index('DCC_ID')
sample_metadata.head()

,Scan.Name,remove,Panel,Roi,Segment,Aoi,Area,Tags,Nuclei,ROICoordinateX,...,NegGeoSD,NegProbes,lib_size,countOfLowEprGene,percentOfLowEprGene,ruv_W1,ruv_W2,ruv_W3,ruv_W4,DCC_ID2
DCC_ID,,,,,,,,,,,,,,,,,,,,,
DSP-1001660008694-G-A02.dcc,"LEAP015, LEAP016",NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,1.0,CD45+,CD45+-aoi-001,7186.20,"PanCK+, CD45+, CD45+",119,76238.40625,...,1.080060,False,22680,0,0.000000,-0.023221,-0.007044,-0.017423,0.022888,DSP-1001660008694-G-A02.dcc
DSP-1001660008694-G-A03.dcc,"LEAP015, LEAP016",NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,1.0,PanCK+,PanCK+-aoi-001,12989.46,"PanCK+, CD45+, PanCK+",239,76238.40625,...,1.329805,False,51774,13492,73.087757,0.008550,-0.022099,0.023774,0.020598,DSP-1001660008694-G-A03.dcc
DSP-1001660008694-G-A05.dcc,"LEAP015, LEAP016",NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,2.0,PanCK+,PanCK+-aoi-001,14576.49,"PanCK+, CD45+, PanCK+",246,75587.60938,...,1.600153,False,77467,11795,63.894908,0.006329,-0.020188,0.015802,-0.001984,DSP-1001660008694-G-A05.dcc
DSP-1001660008694-G-A06.dcc,"LEAP015, LEAP016",NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,3.0,CD45+,CD45+-aoi-001,5066.89,"PanCK+, CD45+, CD45+",85,76252.50000,...,1.114309,False,20049,0,0.000000,-0.035865,-0.006889,-0.027053,0.014727,DSP-1001660008694-G-A06.dcc
DSP-1001660008694-G-A07.dcc,"LEAP015, LEAP016",NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,3.0,PanCK+,PanCK+-aoi-001,41874.55,"PanCK+, CD45+, PanCK+",796,76252.50000,...,1.887786,False,274462,12122,65.666306,0.007790,-0.016314,0.018326,0.004932,DSP-1001660008694-G-A07.dcc


In [12]:
norm_matrix = pd.read_csv(os.path.join(rawdata_path,'sge',"norm_counts.csv"), 
                 sep=";",               # Semicolon separator
                 decimal=",",          # Comma as decimal point
                 quotechar='"',        # Handle quoted strings
                 index_col=0)          # Use the first column as row index (gene name)

norm_expr_matrix = norm_matrix.astype(float)

In [13]:
norm_expr_matrix.head()

,DSP-1001660008694-G-A02.dcc,DSP-1001660008694-G-A03.dcc,DSP-1001660008694-G-A05.dcc,DSP-1001660008694-G-A06.dcc,DSP-1001660008694-G-A07.dcc,DSP-1001660008694-G-A08.dcc,DSP-1001660008694-G-A09.dcc,DSP-1001660008694-G-A10.dcc,DSP-1001660008694-G-A11.dcc,DSP-1001660008694-G-A12.dcc,...,DSP-1001660024022-D-H03.dcc,DSP-1001660024022-D-H04.dcc,DSP-1001660024022-D-H05.dcc,DSP-1001660024022-D-H06.dcc,DSP-1001660024022-D-H07.dcc,DSP-1001660024022-D-H08.dcc,DSP-1001660024022-D-H09.dcc,DSP-1001660024022-D-H10.dcc,DSP-1001660024022-D-H11.dcc,DSP-1001660024022-D-H12.dcc
A2M,8.736860,7.686246,6.830261,7.730451,6.309591,8.198439,7.265407,4.789404,7.753964,8.135174,...,7.983239,6.874335,8.424866,7.618854,7.389477,7.106261,8.151385,7.365071,7.574850,8.325490
ACADM,5.502417,6.386417,7.157426,5.524405,5.528383,4.645522,6.229605,5.613796,5.595107,5.964793,...,5.669373,4.502646,5.404367,6.027267,5.447129,6.175177,5.866024,6.476642,5.846188,5.779037
ACADS,5.095812,5.979818,5.901526,5.018215,5.150049,5.039447,6.287200,4.991735,5.445171,6.055979,...,4.593009,6.173799,5.020409,5.541564,6.217577,5.050971,4.432642,4.116373,3.700987,5.415778
ACAT1,5.551031,7.379331,6.400583,5.686364,7.293440,4.725142,6.379487,4.794026,5.549913,6.974626,...,5.027639,6.449585,6.061562,4.335965,6.585729,5.871773,5.276001,5.155276,5.833425,5.592112
ACVRL1,4.744912,4.843521,6.156299,4.606600,5.220992,5.710566,4.878334,4.835873,3.998791,5.775298,...,4.578111,3.757643,3.932488,6.160220,4.338776,4.984209,4.143997,3.803191,4.515328,5.561657


In [14]:
raw_matrix = pd.read_csv(os.path.join(rawdata_path,'sge',"raw_counts.csv"), 
                 sep=";",               # Semicolon separator
                 decimal=",",          # Comma as decimal point
                 quotechar='"',        # Handle quoted strings
                 index_col=0)          # Use the first column as row index (gene name)

In [15]:

expression_matrix = raw_matrix.astype(float)

In [16]:
expression_matrix.head()

,DSP-1001660008694-G-A02.dcc,DSP-1001660008694-G-A03.dcc,DSP-1001660008694-G-A05.dcc,DSP-1001660008694-G-A06.dcc,DSP-1001660008694-G-A07.dcc,DSP-1001660008694-G-A08.dcc,DSP-1001660008694-G-A09.dcc,DSP-1001660008694-G-A10.dcc,DSP-1001660008694-G-A11.dcc,DSP-1001660008694-G-A12.dcc,...,DSP-1001660024022-D-H03.dcc,DSP-1001660024022-D-H04.dcc,DSP-1001660024022-D-H05.dcc,DSP-1001660024022-D-H06.dcc,DSP-1001660024022-D-H07.dcc,DSP-1001660024022-D-H08.dcc,DSP-1001660024022-D-H09.dcc,DSP-1001660024022-D-H10.dcc,DSP-1001660024022-D-H11.dcc,DSP-1001660024022-D-H12.dcc
A2M,5.0,11.0,12.0,2.0,26.0,12.0,25.0,1.0,53.0,7.0,...,21.0,12.0,314.0,20.0,7.0,52.0,134.0,19.0,59.0,458.0
ACADM,1.0,4.0,12.0,1.0,12.0,1.0,11.0,2.0,10.0,2.0,...,7.0,2.0,39.0,7.0,2.0,35.0,51.0,23.0,26.0,67.0
ACADS,1.0,2.0,3.0,1.0,6.0,1.0,8.0,1.0,6.0,2.0,...,4.0,7.0,28.0,5.0,4.0,20.0,22.0,5.0,5.0,44.0
ACAT1,1.0,9.0,7.0,1.0,45.0,1.0,11.0,1.0,9.0,4.0,...,7.0,9.0,55.0,2.0,5.0,35.0,57.0,15.0,34.0,47.0
ACVRL1,1.0,1.0,4.0,1.0,7.0,2.0,3.0,1.0,2.0,2.0,...,6.0,1.0,11.0,8.0,1.0,21.0,29.0,7.0,14.0,42.0


In [17]:
# Replace '.' with '-' in column names
expression_matrix.columns = expression_matrix.columns.str.replace('.', '-', regex=False)
expression_matrix.columns = expression_matrix.columns.str.replace('-dcc', '.dcc', regex=False)
expression_matrix.columns
# Ensure data is in a usable format
#print(expression_matrix.head())  # Check expression matrix
#expression_matrix.columns

Index(['DSP-1001660008694-G-A02.dcc', 'DSP-1001660008694-G-A03.dcc',
       'DSP-1001660008694-G-A05.dcc', 'DSP-1001660008694-G-A06.dcc',
       'DSP-1001660008694-G-A07.dcc', 'DSP-1001660008694-G-A08.dcc',
       'DSP-1001660008694-G-A09.dcc', 'DSP-1001660008694-G-A10.dcc',
       'DSP-1001660008694-G-A11.dcc', 'DSP-1001660008694-G-A12.dcc',
       ...
       'DSP-1001660024022-D-H03.dcc', 'DSP-1001660024022-D-H04.dcc',
       'DSP-1001660024022-D-H05.dcc', 'DSP-1001660024022-D-H06.dcc',
       'DSP-1001660024022-D-H07.dcc', 'DSP-1001660024022-D-H08.dcc',
       'DSP-1001660024022-D-H09.dcc', 'DSP-1001660024022-D-H10.dcc',
       'DSP-1001660024022-D-H11.dcc', 'DSP-1001660024022-D-H12.dcc'],
      dtype='object', length=5096)

In [18]:
expression_matrix.head()

,DSP-1001660008694-G-A02.dcc,DSP-1001660008694-G-A03.dcc,DSP-1001660008694-G-A05.dcc,DSP-1001660008694-G-A06.dcc,DSP-1001660008694-G-A07.dcc,DSP-1001660008694-G-A08.dcc,DSP-1001660008694-G-A09.dcc,DSP-1001660008694-G-A10.dcc,DSP-1001660008694-G-A11.dcc,DSP-1001660008694-G-A12.dcc,...,DSP-1001660024022-D-H03.dcc,DSP-1001660024022-D-H04.dcc,DSP-1001660024022-D-H05.dcc,DSP-1001660024022-D-H06.dcc,DSP-1001660024022-D-H07.dcc,DSP-1001660024022-D-H08.dcc,DSP-1001660024022-D-H09.dcc,DSP-1001660024022-D-H10.dcc,DSP-1001660024022-D-H11.dcc,DSP-1001660024022-D-H12.dcc
A2M,5.0,11.0,12.0,2.0,26.0,12.0,25.0,1.0,53.0,7.0,...,21.0,12.0,314.0,20.0,7.0,52.0,134.0,19.0,59.0,458.0
ACADM,1.0,4.0,12.0,1.0,12.0,1.0,11.0,2.0,10.0,2.0,...,7.0,2.0,39.0,7.0,2.0,35.0,51.0,23.0,26.0,67.0
ACADS,1.0,2.0,3.0,1.0,6.0,1.0,8.0,1.0,6.0,2.0,...,4.0,7.0,28.0,5.0,4.0,20.0,22.0,5.0,5.0,44.0
ACAT1,1.0,9.0,7.0,1.0,45.0,1.0,11.0,1.0,9.0,4.0,...,7.0,9.0,55.0,2.0,5.0,35.0,57.0,15.0,34.0,47.0
ACVRL1,1.0,1.0,4.0,1.0,7.0,2.0,3.0,1.0,2.0,2.0,...,6.0,1.0,11.0,8.0,1.0,21.0,29.0,7.0,14.0,42.0


## Only use data where we have both the gene expression and the metadata

keep only rows that are not pellets

In [19]:
sample_metadata = sample_metadata[sample_metadata['Sample_Type'] != 'PELLET']

In [20]:
rois_to_use = sorted(set(sample_metadata.index).intersection(set(expression_matrix.columns)))
len(rois_to_use)

4948

In [21]:
rois_to_use

['DSP-1001660008694-G-A02.dcc',
 'DSP-1001660008694-G-A03.dcc',
 'DSP-1001660008694-G-A05.dcc',
 'DSP-1001660008694-G-A06.dcc',
 'DSP-1001660008694-G-A07.dcc',
 'DSP-1001660008694-G-A08.dcc',
 'DSP-1001660008694-G-A09.dcc',
 'DSP-1001660008694-G-A10.dcc',
 'DSP-1001660008694-G-A11.dcc',
 'DSP-1001660008694-G-A12.dcc',
 'DSP-1001660008694-G-B01.dcc',
 'DSP-1001660008694-G-B05.dcc',
 'DSP-1001660008694-G-B06.dcc',
 'DSP-1001660008694-G-B07.dcc',
 'DSP-1001660008694-G-B09.dcc',
 'DSP-1001660008694-G-B11.dcc',
 'DSP-1001660008694-G-B12.dcc',
 'DSP-1001660008694-G-C01.dcc',
 'DSP-1001660008694-G-C02.dcc',
 'DSP-1001660008694-G-C03.dcc',
 'DSP-1001660008694-G-C04.dcc',
 'DSP-1001660008694-G-C05.dcc',
 'DSP-1001660008694-G-C06.dcc',
 'DSP-1001660008694-G-C07.dcc',
 'DSP-1001660008694-G-C08.dcc',
 'DSP-1001660008694-G-C09.dcc',
 'DSP-1001660008694-G-C10.dcc',
 'DSP-1001660008694-G-C11.dcc',
 'DSP-1001660008694-G-C12.dcc',
 'DSP-1001660008694-G-D01.dcc',
 'DSP-1001660008694-G-D02.dcc',
 'DSP-10

In [22]:
sample_metadata = sample_metadata.loc[rois_to_use].sort_index()
expression_matrix = expression_matrix[rois_to_use]
norm_expr_matrix = norm_expr_matrix[rois_to_use]

In [23]:
sample_metadata.shape

(4948, 71)

# Identify hold out test set
need to group by patient


All segments from a patient must go into either train or test.

The test set must contain:

At least one CD45+ and one PanCK+ in Segment

At least one RESECTION and one DIAGNOSTIC in Sample_Type



In [24]:
patient_segment_counts = sample_metadata.groupby(['Patient_ID', 'Segment']).size().unstack(fill_value=0)
patient_segment_counts

Segment,CD45+,Full ROI,PanCK+,PanCK-,PanCK-/CD45-,TLS
Patient_ID,,,,,,
GST15102,2,0,2,0,3,0
GST17605,5,0,5,0,0,0
GST17723,7,0,7,0,5,0
GST18770,4,0,4,0,0,0
GST18928,5,0,5,0,0,0
...,...,...,...,...,...,...
GST48495,46,0,117,26,0,0
GST48581,23,0,21,0,0,0
GST48631,0,0,19,0,0,0


In [25]:
patient_segment_counts[(patient_segment_counts['CD45+'] == 0) & 
                                         (patient_segment_counts['PanCK+'] == 0)]

Segment,CD45+,Full ROI,PanCK+,PanCK-,PanCK-/CD45-,TLS
Patient_ID,,,,,,


In [26]:
patient_summary = sample_metadata.groupby('Patient_ID').agg({
    'Segment': lambda x: set(x),
    'Sample_Type': lambda x: set(x)
})
patient_summary

,Segment,Sample_Type
Patient_ID,,
GST15102,"{PanCK+, PanCK-/CD45-, CD45+}",{RESECTION}
GST17605,"{PanCK+, CD45+}",{RESECTION}
GST17723,"{PanCK+, PanCK-/CD45-, CD45+}",{RESECTION}
GST18770,"{PanCK+, CD45+}",{RESECTION}
GST18928,"{PanCK+, CD45+}",{RESECTION}
...,...,...
GST48495,"{PanCK+, CD45+, PanCK-}","{DIAGNOSTIC, RESECTION}"
GST48581,"{PanCK+, CD45+}",{DIAGNOSTIC}
GST48631,{PanCK+},"{DIAGNOSTIC, RESECTION}"


In [27]:
# Step 1: Get unique Patient_IDs
unique_patients = sample_metadata['Patient_ID'].unique()

# # Step 2: Randomly sample 10% of patients
# np.random.seed(42)
# sampled_patients = np.random.choice(unique_patients,
#                                      size=int(0.1 * len(unique_patients)),
#                                      replace=False)



In [28]:
# # Step 3: Get all rows for those patients
# train_metadata = sample_metadata[ ~sample_metadata['Patient_ID'].isin(sampled_patients)] 
# test_metadata = sample_metadata[sample_metadata['Patient_ID'].isin(sampled_patients)]

these patients were chosen randomly using the code above and used to generate the  data but the seed was not set correctly so the above code changes every time 

In [29]:
train_patients = {'GST15102',
 'GST17605',
 'GST17723',
 'GST18770',
 'GST18928',
 'GST18944',
 'GST19918',
 'GST23088',
 'GST23677',
 'GST24143',
 'GST25373',
 'GST25518',
 'GST26237',
 'GST26311',
 'GST26322',
 'GST26353',
 'GST26423',
 'GST26448',
 'GST26501',
 'GST26791',
 'GST26854',
 'GST26965',
 'GST27131',
 'GST27235',
 'GST27321',
 'GST27337',
 'GST27498',
 'GST27506',
 'GST27925',
 'GST29819',
 'GST30473',
 'GST35861',
 'GST36019',
 'GST36564',
 'GST36713',
 'GST36718',
 'GST36776',
 'GST36789',
 'GST37108',
 'GST37122',
 'GST37411',
 'GST37685',
 'GST37926',
 'GST41084',
 'GST41151',
 'GST43189',
 'GST43338',
 'GST43395',
 'GST43651',
 'GST43654',
 'GST43962',
 'GST44149',
 'GST44189',
 'GST44222',
 'GST44440',
 'GST44790',
 'GST44827',
 'GST46489',
 'GST46735',
 'GST46801',
 'GST47130',
 'GST47289',
 'GST47381',
 'GST47384',
 'GST47471',
 'GST47765',
 'GST47796',
 'GST47859',
 'GST47883',
 'GST48110',
 'GST48134',
 'GST48176',
 'GST48323',
 'GST48454',
 'GST48455',
 'GST48456',
 'GST48495',
 'GST48581',
 'GST48631',
 'GST48798',
 'GST49284'}
len(train_patients)

81

In [30]:
for p in train_patients:
    print(p)

GST26311
GST47384
GST47765
GST41084
GST25518
GST15102
GST36019
GST47289
GST48455
GST46735
GST26501
GST48176
GST43395
GST27498
GST47471
GST25373
GST36718
GST48134
GST23088
GST26353
GST44149
GST26448
GST43962
GST43189
GST44440
GST27321
GST18770
GST37685
GST26854
GST48323
GST18928
GST36776
GST48631
GST23677
GST44827
GST27925
GST29819
GST44189
GST37122
GST46801
GST48798
GST49284
GST17605
GST47859
GST18944
GST26237
GST27131
GST27337
GST27506
GST26423
GST26791
GST36564
GST17723
GST47130
GST48495
GST43654
GST41151
GST48454
GST48456
GST35861
GST30473
GST48110
GST26965
GST47883
GST37411
GST44790
GST26322
GST19918
GST43651
GST47381
GST48581
GST37926
GST36713
GST37108
GST47796
GST36789
GST27235
GST44222
GST24143
GST43338
GST46489


In [31]:
test_patients = {'GST23703',
 'GST25222',
 'GST26416',
 'GST26519',
 'GST26889',
 'GST43294',
 'GST47380',
 'GST47661'}
len(test_patients)

8

In [32]:
train_metadata = sample_metadata[sample_metadata['Patient_ID'].isin(train_patients)]
test_metadata = sample_metadata[sample_metadata['Patient_ID'].isin(test_patients)]

check we have totally separated test and train

In [33]:
(train_metadata.shape[0] + test_metadata.shape[0]) == sample_metadata.shape[0]

True

In [34]:
train_metadata

,Scan.Name,remove,Panel,Roi,Segment,Aoi,Area,Tags,Nuclei,ROICoordinateX,...,NegGeoSD,NegProbes,lib_size,countOfLowEprGene,percentOfLowEprGene,ruv_W1,ruv_W2,ruv_W3,ruv_W4,DCC_ID2
DCC_ID,,,,,,,,,,,,,,,,,,,,,
DSP-1001660008694-G-A02.dcc,"LEAP015, LEAP016",NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,1.0,CD45+,CD45+-aoi-001,7186.20,"PanCK+, CD45+, CD45+",119,76238.40625,...,1.080060,False,22680,0,0.000000,-0.023221,-0.007044,-0.017423,0.022888,DSP-1001660008694-G-A02.dcc
DSP-1001660008694-G-A03.dcc,"LEAP015, LEAP016",NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,1.0,PanCK+,PanCK+-aoi-001,12989.46,"PanCK+, CD45+, PanCK+",239,76238.40625,...,1.329805,False,51774,13492,73.087757,0.008550,-0.022099,0.023774,0.020598,DSP-1001660008694-G-A03.dcc
DSP-1001660008694-G-A05.dcc,"LEAP015, LEAP016",NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,2.0,PanCK+,PanCK+-aoi-001,14576.49,"PanCK+, CD45+, PanCK+",246,75587.60938,...,1.600153,False,77467,11795,63.894908,0.006329,-0.020188,0.015802,-0.001984,DSP-1001660008694-G-A05.dcc
DSP-1001660008694-G-A06.dcc,"LEAP015, LEAP016",NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,3.0,CD45+,CD45+-aoi-001,5066.89,"PanCK+, CD45+, CD45+",85,76252.50000,...,1.114309,False,20049,0,0.000000,-0.035865,-0.006889,-0.027053,0.014727,DSP-1001660008694-G-A06.dcc
DSP-1001660008694-G-A07.dcc,"LEAP015, LEAP016",NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,3.0,PanCK+,PanCK+-aoi-001,41874.55,"PanCK+, CD45+, PanCK+",796,76252.50000,...,1.887786,False,274462,12122,65.666306,0.007790,-0.016314,0.018326,0.004932,DSP-1001660008694-G-A07.dcc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DSP-1001660024022-D-H08.dcc,LEAP144_LEAP145_LEAP146,NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,36.0,Full ROI,Full ROI-aoi-001,201930.07,NaN,617,55471.23828,...,1.632776,False,703607,15324,83.011918,0.012052,0.000509,0.005955,0.050203,DSP-1001660024022-D-H08.dcc
DSP-1001660024022-D-H09.dcc,LEAP144_LEAP145_LEAP146,NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,37.0,CD45+,CD45+-aoi-001,35826.09,"CD45+, CD45+",652,55471.23828,...,1.506462,False,1876428,17190,93.120260,0.024626,-0.021292,0.029350,0.127766,DSP-1001660024022-D-H09.dcc
DSP-1001660024022-D-H10.dcc,LEAP144_LEAP145_LEAP146,NaN,(v1.0) Human NGS Whole Transcriptome Atlas RNA,38.0,CD45+,CD45+-aoi-001,18974.28,"CD45+, CD45+",353,56662.30078,...,1.762112,False,561625,17227,93.320693,0.018843,-0.019651,0.035147,0.131360,DSP-1001660024022-D-H10.dcc


In [35]:
rois_train = sorted(set(train_metadata.index))
rois_test = sorted(set(test_metadata.index))

In [36]:
print(f"train rois: {len(rois_train)}")
print(f"test rois: {len(rois_test)}")
print(f"{len(rois_test)/len(rois_train):%}")

train rois: 4503
test rois: 445
9.882301%


In [37]:
(len(rois_test)+len(rois_train))==len(rois_to_use)

True

In [38]:
expression_train = expression_matrix[rois_train]
expression_test = expression_matrix[rois_test]

In [39]:
print("Original Sample_Type proportions:")
print(sample_metadata['Sample_Type'].value_counts(normalize=True))

print("\nSampled Sample_Type proportions:")
print(train_metadata['Sample_Type'].value_counts(normalize=True))
print("\nSampled Sample_Type proportions:")
print(test_metadata['Sample_Type'].value_counts(normalize=True))

Original Sample_Type proportions:
Sample_Type
DIAGNOSTIC    0.811439
RESECTION     0.159256
Pre-NACT      0.015562
1_1           0.010509
2_2           0.003234
Name: proportion, dtype: float64

Sampled Sample_Type proportions:
Sample_Type
DIAGNOSTIC    0.827448
RESECTION     0.140351
Pre-NACT      0.017100
1_1           0.011548
2_2           0.003553
Name: proportion, dtype: float64

Sampled Sample_Type proportions:
Sample_Type
DIAGNOSTIC    0.649438
RESECTION     0.350562
Name: proportion, dtype: float64


In [40]:
expression_test.shape

(18460, 445)

In [41]:
test_metadata.shape

(445, 71)

# Gene Expression

In [42]:
sample_metadata = sample_metadata.sort_index()
test_metadata = test_metadata.sort_index()
train_metadata = train_metadata.sort_index()

In [43]:
rois_cd45_train = list(train_metadata[train_metadata['Segment']=='CD45+'].index)
rois_panck_train = list(train_metadata[train_metadata['Segment']=='PanCK+'].index)

In [44]:
rois_cd45_test = list(test_metadata[test_metadata['Segment']=='CD45+'].index)
rois_panck_test = list(test_metadata[test_metadata['Segment']=='PanCK+'].index)

In [45]:
log_transformed_expr = expression_matrix.apply(np.log1p).T

In [46]:

#final_expression_matrix = expression_matrix.T.astype(float).sort_index()
#log_transformed_expr = final_expression_matrix.apply(np.log1p)
norm_expr_matrix = norm_expr_matrix.T.astype(float).sort_index()
norm_expr_matrix.head()

,A2M,ACADM,ACADS,ACAT1,ACVRL1,PSEN1,ADA,SGCA,ADRB2,ADRB3,...,TNFSF18,CCL23,WNT6,VEGFD,GLI1,NDUFA3,HAMP,ADH1B,CCL3,CST2
DSP-1001660008694-G-A02.dcc,8.736860,5.502417,5.095812,5.551031,4.744912,5.276005,5.327490,4.709154,4.748537,4.524060,...,4.857875,4.948848,5.321573,5.291184,4.677396,5.727129,4.605154,4.636586,5.203187,4.491745
DSP-1001660008694-G-A03.dcc,7.686246,6.386417,5.979818,7.379331,4.843521,6.654290,4.760808,5.580089,4.966828,4.663518,...,4.584349,5.137348,5.168634,4.940911,5.690619,6.918611,4.605043,4.866588,5.595081,4.665243
DSP-1001660008694-G-A05.dcc,6.830261,7.157426,5.901526,6.400583,6.156299,5.308238,6.423473,5.900474,4.529480,4.094271,...,3.912445,5.333651,5.264247,6.460309,5.582917,6.148312,4.851412,4.275327,6.059923,5.115517
DSP-1001660008694-G-A06.dcc,7.730451,5.524405,5.018215,5.686364,4.606600,5.247726,5.229229,4.408976,4.577911,4.159722,...,4.524535,4.641437,5.110279,5.151759,4.355345,5.772854,4.260822,4.277445,4.997191,4.287743
DSP-1001660008694-G-A07.dcc,6.309591,5.528383,5.150049,7.293440,5.220992,5.758353,5.327022,4.846551,3.283847,5.352596,...,3.451089,3.901585,4.276544,4.759025,4.926420,6.396389,4.545595,4.159937,5.266447,4.925050


## Remove Contamination

Tumour segments have contamination from plasma cells so we need to remove these genes before doing our analysis

make sure we are only doing this for PanCK+ AOIs

In [47]:
tumour_aois = sorted(rois_panck_test + rois_panck_train)

len(tumour_aois)

2374

In [48]:
plasma_genes = ["IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHM", "IGHA1", "IGKC", "IGLL5"]

In [49]:
tumour_expr = norm_expr_matrix.loc[tumour_aois,:]
tumour_expr.head()
#tumour_expr.loc[plasma_genes,:].head()

,A2M,ACADM,ACADS,ACAT1,ACVRL1,PSEN1,ADA,SGCA,ADRB2,ADRB3,...,TNFSF18,CCL23,WNT6,VEGFD,GLI1,NDUFA3,HAMP,ADH1B,CCL3,CST2
DSP-1001660008694-G-A03.dcc,7.686246,6.386417,5.979818,7.379331,4.843521,6.654290,4.760808,5.580089,4.966828,4.663518,...,4.584349,5.137348,5.168634,4.940911,5.690619,6.918611,4.605043,4.866588,5.595081,4.665243
DSP-1001660008694-G-A05.dcc,6.830261,7.157426,5.901526,6.400583,6.156299,5.308238,6.423473,5.900474,4.529480,4.094271,...,3.912445,5.333651,5.264247,6.460309,5.582917,6.148312,4.851412,4.275327,6.059923,5.115517
DSP-1001660008694-G-A07.dcc,6.309591,5.528383,5.150049,7.293440,5.220992,5.758353,5.327022,4.846551,3.283847,5.352596,...,3.451089,3.901585,4.276544,4.759025,4.926420,6.396389,4.545595,4.159937,5.266447,4.925050
DSP-1001660008694-G-A09.dcc,7.265407,6.229605,6.287200,6.379487,4.878334,4.933541,5.575112,5.491822,6.108288,4.448252,...,3.826982,4.350966,5.731811,5.006641,4.079268,6.575408,5.051638,5.732952,5.391057,5.649282
DSP-1001660008694-G-A11.dcc,7.753964,5.595107,5.445171,5.549913,3.998791,4.752077,6.448923,4.563710,4.936684,5.173806,...,4.609001,4.857005,5.331441,5.662894,4.226408,6.211358,5.106878,4.272509,4.452907,5.884296


check correlation of all genes with known plasma genes to identify other genes that are likely from plasma cells rather than tumour epithelial 

In [50]:
# compute correlations of every gene vs. the mean plasma score
plasma_score = tumour_expr[plasma_genes].mean(axis=1)

corrs = tumour_expr.corrwith(plasma_score, axis=0)  # correlation of each gene with plasma_score
corrs = corrs.sort_values(ascending=False)

# turn into dataframe
corr_df = pd.DataFrame({
    "Gene": corrs.index,
    "Correlation_with_Plasma": corrs.values
})

Inspect correlation and remove anything over 0.7

In [51]:
corr_df[corr_df['Correlation_with_Plasma']>0.7]

,Gene,Correlation_with_Plasma
0,IGHG3,0.976167
1,IGHG1,0.973902
2,IGHG4,0.973033
3,IGHG2,0.972980
4,IGKC,0.969238
5,IGLL5,0.896662
6,IGHM,0.751535
7,IGHA1,0.725487


In [52]:
contamination_genes = corr_df.loc[corr_df['Correlation_with_Plasma']>0.7,'Gene'].to_list()

we should consider anything over 0.5 as possible plasma - JCHAIN and MZB1 are definite plasma genes

In [53]:
cond = (corr_df['Correlation_with_Plasma']<=0.7)&(corr_df['Correlation_with_Plasma']>0.5)
corr_df[cond]

,Gene,Correlation_with_Plasma
8,JCHAIN,0.600773
9,MZB1,0.584040


In [54]:
contamination_genes = contamination_genes + corr_df.loc[cond,'Gene'].to_list()
contamination_genes

['IGHG3',
 'IGHG1',
 'IGHG4',
 'IGHG2',
 'IGKC',
 'IGLL5',
 'IGHM',
 'IGHA1',
 'JCHAIN',
 'MZB1']

In [55]:
cond = (corr_df['Correlation_with_Plasma']<=0.5)&(corr_df['Correlation_with_Plasma']>0.4)
corr_df[cond]

,Gene,Correlation_with_Plasma
10,LYZ,0.478247
11,DERL3,0.476075
12,CD74,0.469618
13,CD79A,0.460437
14,TRBC1,0.456635
15,TENT5C,0.454237
16,C1QB,0.449500
17,C1QC,0.448308
18,HLA-DPA1,0.448198
19,PIM2,0.447166


Strongly immune-specific (should consider removing as we only want tumour-epithelial)

- CD79A (corr 0.46) → B-cell receptor, B lineage only.
- TRBC1 (corr 0.46) → T-cell receptor β chain, T lineage only.
- C1QA, C1QB, C1QC (corr 0.40, 0.45, 0.45) → complement genes, macrophage/myeloid lineage.
- ADA2 (corr 0.43) → myeloid/monocyte.
- LYZ (corr 0.48) → lysozyme, classic macrophage/neutrophil marker.
- IL10RA (corr 0.42) → monocyte/macrophage, not epithelial.
- EVI2B, CORO1A, WIPF1 (corr 0.42, 0.41, 0.41) → leukocyte cytoskeleton/trafficking, T/NK/myeloid associated.

## Training Set Only
Need to determine genes based on train set to avoid bias

In [56]:
TRAIN_SET_ONLY = True

In [57]:
final_norm_matrix = norm_expr_matrix.copy()

In [58]:
if TRAIN_SET_ONLY:
    final_norm_matrix = final_norm_matrix.loc[rois_train]
    train_tumour_aois = set(tumour_aois).intersection(set(rois_train))
else:
    train_tumour_aois = tumour_aois

set expression to zero for contamination genes in tumour epithelial

In [59]:
final_norm_matrix.loc[list(train_tumour_aois),contamination_genes] = 0
# check we have only set to 0 on the PanCK+ AOIs
final_norm_matrix.loc[:,contamination_genes]

,IGHG3,IGHG1,IGHG4,IGHG2,IGKC,IGLL5,IGHM,IGHA1,JCHAIN,MZB1
DSP-1001660008694-G-A02.dcc,6.321806,6.486105,7.910064,6.526511,6.411107,5.731868,6.819770,5.011164,5.068021,5.381198
DSP-1001660008694-G-A03.dcc,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
DSP-1001660008694-G-A05.dcc,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
DSP-1001660008694-G-A06.dcc,7.993664,8.253587,8.136558,8.223948,9.065838,7.226121,6.803185,5.836918,5.588335,6.079026
DSP-1001660008694-G-A07.dcc,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
DSP-1001660024022-D-H08.dcc,12.405782,12.672189,13.227088,12.129353,12.467595,10.147085,7.915245,7.571767,5.479322,7.427696
DSP-1001660024022-D-H09.dcc,7.342027,7.469779,7.941272,7.368211,7.414136,4.386362,4.661535,3.905513,3.287045,4.385806
DSP-1001660024022-D-H10.dcc,7.130658,7.233049,7.577838,7.171591,7.478592,4.059951,4.348946,0.534741,2.254485,4.497559
DSP-1001660024022-D-H11.dcc,11.400145,11.588678,12.208536,11.288754,11.512960,8.785918,6.600135,6.661187,4.583878,6.710718


## Setup AnnData object

use the normalised, batch-corrected data to find the HVGs and SVGs

In [60]:
import scanpy as sc
import squidpy as sq
from anndata import AnnData

c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


final_norm_matrix
sample_metadata

In [61]:
if TRAIN_SET_ONLY:
    meta_to_use = train_metadata.sort_index().copy()
else:
    meta_to_use = sample_metadata.sort_index().copy()


In [62]:
# final_norm_matrix = final_norm_matrix.sort_index().sort_index(axis=1)
# meta_to_use = meta_to_use.sort_index()



# adata.obsm["spatial"] = meta_to_use[["ROICoordinateX", "ROICoordinateY"]].to_numpy()
# adata.obs["LEAPID"] = meta_to_use["LEAP_ID"].astype(str).to_numpy()
# adata.obs["segment"] = meta_to_use["Segment"].astype(str).to_numpy()

# adata = adata[sorted(adata.obs_names), sorted(adata.var_names)].copy()

In [63]:
# Create AnnData object
#adata = AnnData(final_norm_matrix) #expression matrix transposed
adata = AnnData(final_norm_matrix.loc[meta_to_use.index].copy())

# adata.obsm["spatial"] = sample_metadata[['ROICoordinateX', 'ROICoordinateY']].to_numpy()  #roi coords
# adata.obs["LEAPID"] = sample_metadata['LEAP_ID']
# adata.obs["segment"] = sample_metadata['Segment']

adata.obsm["spatial"] = meta_to_use[['ROICoordinateX', 'ROICoordinateY']].to_numpy()  #roi coords
adata.obs["LEAPID"] = meta_to_use['LEAP_ID']
adata.obs["segment"] = meta_to_use['Segment']

adata.X = np.nan_to_num(adata.X, nan=0.0, posinf=0.0, neginf=0.0)
adata = adata[sorted(adata.obs_names), sorted(adata.var_names)].copy()

## Highly Variable Genes (HVGs)

In [64]:
from collections import defaultdict

# Dictionary to store HVG results per segment
hvg_results_by_segment = {}

# Loop over each segment type
for segment in ['CD45+','PanCK+']:
    print(f"Processing segment: {segment}")
    
    # Subset AnnData
    adata_sub = adata[adata.obs["segment"] == segment].copy()
    
    # Filter out genes with all-zero expression (optional but good)
    sc.pp.filter_genes(adata_sub, min_cells=1)
    
    # Identify highly variable genes
    sc.pp.highly_variable_genes(
        adata_sub,
        flavor='cell_ranger',
        n_top_genes=100
    )
    
    # Store DataFrame of HVGs
    hvg_df = adata_sub.var[adata_sub.var["highly_variable"]].copy()
    hvg_results_by_segment[segment] = hvg_df

    print(f"Found {hvg_df.shape[0]} highly variable genes for {segment}")


Processing segment: CD45+
Found 100 highly variable genes for CD45+
Processing segment: PanCK+
Found 100 highly variable genes for PanCK+


Get top 50 HVGs per segment

In [65]:
HVGs_cd45 = list(hvg_results_by_segment['CD45+'].sort_values(by='dispersions_norm', ascending=False)[:50].index)
HVGs_cd45

['FDCSP',
 'KRT14',
 'MUCL1',
 'IGLL5',
 'APOD',
 'IGHG1',
 'S100A7',
 'IGHA1',
 'IGHG4',
 'IGHG2',
 'IGHM',
 'IGKC',
 'KRT23',
 'GABRP',
 'MGP',
 'IGHG3',
 'ACTG2',
 'CRABP1',
 'LCN2',
 'KRT5',
 'S100A8',
 'FABP7',
 'ZFP57',
 'MMP7',
 'ELN',
 'S100A1',
 'SERPINA3',
 'CRYAB',
 'TRPM3',
 'ELF5',
 'CPB1',
 'KRT17',
 'AARD',
 'MZT2B',
 'TPSAB1',
 'CCL24',
 'C1QL2',
 'HLA-DQA1',
 'FNTB',
 'SFRP1',
 'KRT15',
 'PPP2R2B',
 'RPL10L',
 'KRT6A',
 'SMR3A',
 'AZGP1',
 'AREG',
 'FXYD3',
 'JCHAIN',
 'SPP1']

For tumour (PanCK+) segments we need to filter out any immune genes from plasma cells that have contaminated the segment, otherwise these will dominate the analyses

In [66]:
hvg_results_by_segment['PanCK+'].index

Index(['A2M', 'AARD', 'ACTA2', 'ACTG2', 'ANK1', 'ANKRD30A', 'APOD', 'AQP3',
       'AZGP1', 'C1S', 'C4B', 'CA9', 'CALML3', 'CALML5', 'CASP14', 'CCL28',
       'CCND1', 'CFB', 'CHI3L1', 'CLCA2', 'COL3A1', 'CP', 'CPB1', 'CRABP1',
       'CRYAB', 'CSN3', 'CXCL1', 'CXCL6', 'CXCL8', 'CYP1B1', 'DSG1', 'ECEL1',
       'EFEMP1', 'ELN', 'EN1', 'FABP7', 'FDCSP', 'FGFBP2', 'FN1', 'FNTB',
       'FOS', 'FOXA1', 'GABRP', 'GLYATL2', 'HLA-DQA1', 'HLA-DQB1', 'IDO1',
       'IFI27', 'IRS4', 'IRX1', 'KLK5', 'KLK6', 'KLK7', 'KLK8', 'KRT14',
       'KRT15', 'KRT16', 'KRT17', 'KRT23', 'KRT5', 'KRT6A', 'KRT81', 'LBP',
       'LCN2', 'LTF', 'MGP', 'MMP7', 'MSLN', 'MUCL1', 'MZT2B', 'NCCRP1',
       'NDRG2', 'ORM1', 'PCSK1N', 'PEG10', 'PI3', 'PIGR', 'RARRES1', 'S100A1',
       'S100A2', 'S100A7', 'S100A8', 'S100A9', 'S100P', 'SCGB2A2', 'SELENOP',
       'SERPINA3', 'SERPINB5', 'SOX10', 'SPIB', 'STAC2', 'SYT8', 'TFF3',
       'TRIM29', 'TRPM3', 'TUBB2B', 'UBD', 'UCHL1', 'WFDC2', 'ZFP57'],
      dtype='object')

Check none of the contamination genes made it into the HVGs

In [67]:
hvg_results_by_segment['PanCK+'].index.isin(contamination_genes).any()

False

Check whether the borderline plasma-correlated genes are in our list of HVGs

In [68]:
poss_genes = ['CD79A','TRBC1','C1QA','C1QB','C1QC','ADA2','LYZ','IL10RA','EVI2B','CORO1A','WIPF1']
hvg_results_by_segment['PanCK+'].index.isin(poss_genes).any()

False

In [69]:
HVGs_panck = list(hvg_results_by_segment['PanCK+'].sort_values(by='dispersions_norm', ascending=False)[:50].index)
HVGs_panck

['MUCL1',
 'FDCSP',
 'KRT14',
 'APOD',
 'S100A8',
 'S100A7',
 'ACTG2',
 'MGP',
 'AQP3',
 'CPB1',
 'S100A9',
 'S100P',
 'CSN3',
 'ELN',
 'KRT16',
 'FABP7',
 'S100A2',
 'LCN2',
 'CRABP1',
 'IRS4',
 'KRT6A',
 'HLA-DQA1',
 'MMP7',
 'AARD',
 'KLK5',
 'RARRES1',
 'CXCL1',
 'PIGR',
 'CASP14',
 'UBD',
 'KRT17',
 'KRT81',
 'SERPINA3',
 'CHI3L1',
 'CCL28',
 'CA9',
 'GABRP',
 'SCGB2A2',
 'KRT23',
 'CXCL8',
 'KRT5',
 'IRX1',
 'LTF',
 'PI3',
 'C4B',
 'EFEMP1',
 'STAC2',
 'FNTB',
 'CYP1B1',
 'CALML3']

In [70]:
from collections import defaultdict

top_n = 100
segment_to_top_genes = {
    segment: set(hvg_df.sort_values("dispersions_norm", ascending=False).index[:top_n])
    for segment, hvg_df in hvg_results_by_segment.items()
}

# Count how many segments each gene appears in
from collections import Counter

gene_counter = Counter()
for genes in segment_to_top_genes.values():
    gene_counter.update(genes)

# Show most common HVGs
print("Most frequently variable genes across segments:")
for gene, count in gene_counter.most_common(10):
    print(f"{gene}: {count} segments")

Most frequently variable genes across segments:
AZGP1: 2 segments
FDCSP: 2 segments
CRABP1: 2 segments
ZFP57: 2 segments
RARRES1: 2 segments
APOD: 2 segments
TRPM3: 2 segments
SPIB: 2 segments
IRX1: 2 segments
S100A1: 2 segments


use the log transformed raw expression for the ground truth once we know what the HVGs are

In [71]:
hvgs_df_panck = log_transformed_expr.loc[rois_panck_train,HVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
hvgs_df_cd45 = log_transformed_expr.loc[rois_cd45_train,HVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [72]:

###hvgs_df_panck = expression_matrix.T.loc[rois_panck,HVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
###hvgs_df_cd45 = expression_matrix.T.loc[rois_cd45,HVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [73]:
hvgs_df_panck.to_csv(os.path.join(outcome_path,f"hvg_panck_raw.csv"), index=False)
hvgs_df_cd45.to_csv(os.path.join(outcome_path,f"hvg_cd45_raw.csv"), index=False)

TEST SET

In [74]:
hvgs_df_panck = log_transformed_expr.loc[rois_panck_test,HVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
hvgs_df_cd45 = log_transformed_expr.loc[rois_cd45_test,HVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})
hvgs_df_panck.to_csv(os.path.join(outcome_path,f"hvg_panck_raw_TEST.csv"), index=False)
hvgs_df_cd45.to_csv(os.path.join(outcome_path,f"hvg_cd45_raw_TEST.csv"), index=False)

## Spatially Variable Genes (SVGs)

In [75]:
### ORIGINAL CODE

# results = {}

# for leapid in adata.obs["LEAPID"].unique():
#     for segment in ['CD45+','PanCK+']:
#         print(segment)
#         subset = adata[
#             (adata.obs["LEAPID"] == leapid) &
#             (adata.obs["segment"] == segment)
#         ].copy()
        

#         # Compute spatial graph and spatial autocorrelation
#         n_obs = subset.n_obs


#         if n_obs > 1:
#             print(leapid, segment, n_obs)
#             #print(f"{leapid} {segment} {n_obs}")
#             n_neighbors = min(5, n_obs - 1)


#             # Build spatial graph and compute Moran’s I
#             sq.gr.spatial_neighbors(subset, coord_type="generic", n_neighs=n_neighbors)
#             sq.gr.spatial_autocorr(subset, mode="moran")

#             # Extract top genes
#             moranI = subset.uns['moranI']
#             top_genes = moranI.sort_values("I", ascending=False).head(10)
            
#             results[(leapid, segment)] = top_genes
#             #print(f"Top genes for {leapid}-{segment}: {top_genes.index.tolist()}")

In [76]:


# # Dictionary: segment → Counter of gene frequencies
# gene_counts_by_segment = defaultdict(Counter)

# top_n = 20  # Adjust based on how many top genes you consider

# for (leapid, segment), df in results.items():
#     top_genes = df.head(top_n).index.tolist()
#     gene_counts_by_segment[segment].update(top_genes)

LEAP115
LEAP130
LEAP012
LEAP106
LEAP028

In [77]:
# test_leapid = "LEAP012"
# test_segment = "CD45+"
# print(f"{test_leapid} run 1")
# subset = adata[
#     (adata.obs["LEAPID"] == test_leapid) &
#     (adata.obs["segment"] == test_segment)
# ].copy()

# sq.gr.spatial_neighbors(
#     subset,
#     coord_type="generic",
#     n_neighs=min(5, subset.n_obs - 1)
# )

# sq.gr.spatial_autocorr(
#     subset,
#     mode="moran",
#     n_perms=None,
#     n_jobs=1,
#     seed=42
# )

# subset.uns["moranI"].sort_values("I", ascending=False).head(20)

In [78]:

# print(f"{test_leapid} run 2")
# subset = adata[
#     (adata.obs["LEAPID"] == test_leapid) &
#     (adata.obs["segment"] == test_segment)
# ].copy()

# sq.gr.spatial_neighbors(
#     subset,
#     coord_type="generic",
#     n_neighs=min(5, subset.n_obs - 1)
# )

# sq.gr.spatial_autocorr(
#     subset,
#     mode="moran",
#     n_perms=None,
#     n_jobs=1,
#     seed=42
# )

# subset.uns["moranI"].sort_values("I", ascending=False).head(20)

### main loop

In [79]:
import random
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
results = {}
segments = ["CD45+", "PanCK+"]

# Calculate Moran's I within each LEAPID × segment
for leapid in adata.obs["LEAPID"].unique():
    for segment in segments:
        
        subset = adata[
            (adata.obs["LEAPID"] == leapid) &
            (adata.obs["segment"] == segment)
        ].copy()

        # Force deterministic observation order
        subset = subset[sorted(subset.obs_names)].copy()

        n_obs = subset.n_obs
        # Skip tiny or degenerate spatial graphs
        if n_obs < 6:
            print("skipping ",leapid)
            continue


        print(leapid, segment, n_obs)

        n_neighbors = min(5, n_obs - 1)
        print("1")
        sq.gr.spatial_neighbors(
            subset,
            coord_type="generic",
            n_neighs=n_neighbors
        )


        sq.gr.spatial_autocorr(
            subset,
            mode="moran",
            n_perms=None,
            n_jobs=1,
            seed=SEED
        )
        moranI = subset.uns["moranI"].copy()

        moranI = moranI.dropna(subset=["I"])

        if "var_norm" in moranI.columns:
            moranI = moranI[moranI["var_norm"] > 0]

        # Keep positive spatial autocorrelation only
        moranI = moranI[moranI["I"] > 0]

        # Deterministic ranking with gene-name tie break
        moranI = (
            moranI
            .reset_index(names="gene")
            .sort_values(
                ["I", "gene"],
                ascending=[False, True],
                kind="mergesort"
            )
            .set_index("gene")
        )

        results[(leapid, segment)] = moranI


LEAP015 CD45+ 10
1
LEAP015 PanCK+ 14
1
skipping  LEAP016
LEAP016 PanCK+ 7
1
LEAP017 CD45+ 8
1
LEAP017 PanCK+ 23
1


c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(


LEAP002 CD45+ 8
1
LEAP002 PanCK+ 8
1
skipping  LEAP006
skipping  LEAP006
LEAP005 CD45+ 20
1


c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(


LEAP005 PanCK+ 19
1
LEAP034 CD45+ 26
1
LEAP034 PanCK+ 36
1
LEAP004 CD45+ 12
1


c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(


... (88 outputs left out)

LEAP072 PanCK+ 6
1
LEAP142 CD45+ 7
1
LEAP142 PanCK+ 10
1
skipping  LEAP143
LEAP143 PanCK+ 8
1


c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\squidpy\gr\_ppatterns.py:553: RuntimeWarning: invalid value encountered in divide
  z_norm = (score - params["expected"]) / seScore_norm
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(


LEAP144 CD45+ 15
1
LEAP144 PanCK+ 21
1
LEAP145 CD45+ 12
1
LEAP145 PanCK+ 6
1


c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\scanpy\metrics\_common.py:62: UserWarning: 10 variables were constant, will return nan for these.
  warnings.warn(
c:\Users\hooll\anaconda3\envs\multi3\Lib\site-packages\squidpy\gr\_ppatterns.py:553: RuntimeWarning: invalid value encountered in divide
  z_norm = (score - params["expected"]) / seScore_norm


In [80]:
print(len(results))
print(sorted(results.keys())[:20])

197
[('LEAP001', 'CD45+'), ('LEAP001', 'PanCK+'), ('LEAP002', 'CD45+'), ('LEAP002', 'PanCK+'), ('LEAP003', 'CD45+'), ('LEAP003', 'PanCK+'), ('LEAP004', 'CD45+'), ('LEAP004', 'PanCK+'), ('LEAP005', 'CD45+'), ('LEAP005', 'PanCK+'), ('LEAP008', 'CD45+'), ('LEAP008', 'PanCK+'), ('LEAP009', 'CD45+'), ('LEAP009', 'PanCK+'), ('LEAP010', 'CD45+'), ('LEAP010', 'PanCK+'), ('LEAP011', 'PanCK+'), ('LEAP012', 'PanCK+'), ('LEAP013', 'CD45+'), ('LEAP013', 'PanCK+')]


### Count

In [81]:
combined_df = pd.concat(
    [
        df.assign(LEAPID=leapid, segment=segment)
        for (leapid, segment), df in results.items()
    ]
).reset_index(names="gene")

In [82]:
combined_df

,gene,I,pval_norm,var_norm,pval_norm_fdr_bh,LEAPID,segment
0,NLGN3,0.497059,4.124067e-08,0.012866,0.000409,LEAP015,CD45+
1,CST7,0.495561,4.436488e-08,0.012866,0.000409,LEAP015,CD45+
2,TMEM259,0.445482,4.625910e-07,0.012866,0.002846,LEAP015,CD45+
3,SMIM10L2A,0.437969,6.469625e-07,0.012866,0.002986,LEAP015,CD45+
4,MYO1G,0.419788,1.431604e-06,0.012866,0.004939,LEAP015,CD45+
...,...,...,...,...,...,...,...
1078282,MRPL36,0.000149,1.884187e-01,0.010617,0.437886,LEAP145,CD45+
1078283,SNX24,0.000128,1.884744e-01,0.010617,0.437886,LEAP145,CD45+
1078284,CLCNKB,0.000123,1.884876e-01,0.010617,0.437886,LEAP145,CD45+
1078285,RFC1,0.000072,1.886206e-01,0.010617,0.437886,LEAP145,CD45+


In [83]:
#combined_df.to_csv(os.path.join(outcome_path,"svg_summary.csv"), index=False)

In [84]:
from collections import defaultdict, Counter

# Aggregate recurrent SVGs across LEAPIDs
top_n_per_leapid = 20


gene_scores_by_segment = defaultdict(list)

for (leapid, segment), moranI in results.items():
    top = moranI.head(top_n_per_leapid)

    for gene, row in top.iterrows():
        gene_scores_by_segment[segment].append({
            "gene": gene,
            "LEAPID": leapid,
            "I": row["I"]
        })


In [85]:
min_frequency = 2
min_mean_I = 0.1
n_final_genes = 50

svg_summary = {}

for segment, records in gene_scores_by_segment.items():
    tmp = pd.DataFrame(records)

    summary = (
        tmp.groupby("gene")
        .agg(
            frequency=("gene", "size"),
            mean_I=("I", "mean"),
            median_I=("I", "median"),
            max_I=("I", "max")
        )
    )

    summary = summary[
        (summary["frequency"] >= min_frequency) &
        (summary["mean_I"] > min_mean_I) &
        (summary["median_I"] > 0)
    ]

    summary = summary.sort_values(
        ["frequency", "median_I", "mean_I", "max_I"],
        ascending=[False, False, False, False],
        kind="mergesort"
    )

    svg_summary[segment] = summary


SVGs_cd45 = svg_summary["CD45+"].head(n_final_genes).index.tolist()
SVGs_panck = svg_summary["PanCK+"].head(n_final_genes).index.tolist()

Get top 50 SVGs per segment

In [86]:
# SVGs_cd45 = [gene for segment, counter in gene_counts_by_segment.items() for gene,_ in counter.most_common(50) if segment=='CD45+']
# SVGs_cd45
# SVGs_panck = [gene for segment, counter in gene_counts_by_segment.items() for gene,_ in counter.most_common(50) if segment=='PanCK+']
# SVGs_panck

In [87]:
n_final_genes = 50
SVGs_cd45 = svg_summary["CD45+"].head(n_final_genes).index.tolist()
SVGs_panck = svg_summary["PanCK+"].head(n_final_genes).index.tolist()


In [88]:
svg_summary["CD45+"].head(n_final_genes)

,frequency,mean_I,median_I,max_I
gene,,,,
SUPT7L,7,0.480083,0.481808,0.716156
MGP,6,0.624134,0.650785,0.866975
COL1A1,6,0.520196,0.516377,0.733422
S100A9,5,0.489549,0.518901,0.608382
IFI6,5,0.474314,0.474921,0.523610
RPS17,4,0.650031,0.651635,0.840271
PITX1,4,0.615040,0.637347,0.775934
RPL26,4,0.605877,0.623479,0.745787
PTBP1,4,0.609254,0.595305,0.797121


In [89]:
svg_summary["PanCK+"].head(n_final_genes)

,frequency,mean_I,median_I,max_I
gene,,,,
ADAM15,7,0.665979,0.604776,0.857836
TMEM106C,6,0.772289,0.799011,0.872954
SUPT7L,6,0.629905,0.572374,0.863941
PTBP1,5,0.669381,0.702762,0.908343
HLA-B,5,0.637081,0.671572,0.825700
CRABP2,5,0.686992,0.622921,0.914537
ENO1,5,0.571757,0.590782,0.694892
RPL12,5,0.582263,0.569874,0.643044
HLA-DPA1,4,0.662820,0.680463,0.786466


In [90]:
#SVGs_cd45_run1  = SVGs_cd45.copy()
#SVGs_panck_run1  = SVGs_panck.copy()

In [91]:
# sorted(SVGs_cd45_run1) == sorted(SVGs_cd45)
# sorted(SVGs_panck_run1) == sorted(SVGs_panck)

In [92]:
SVGs_cd45 = sorted(SVGs_cd45)
SVGs_cd45

['ADAM15',
 'AEBP1',
 'APOE',
 'BGN',
 'BST2',
 'CLU',
 'COL1A1',
 'COL3A1',
 'COTL1',
 'CRABP2',
 'EIF4A2',
 'EYA2',
 'FDCSP',
 'FN1',
 'GPNMB',
 'HLA-B',
 'HSPA8',
 'IFI44',
 'IFI44L',
 'IFI6',
 'IGHG1',
 'IGHG2',
 'IGHG3',
 'IGHG4',
 'IGKC',
 'ISG15',
 'KRT7',
 'LAPTM4B',
 'LGALS1',
 'MGP',
 'MMP1',
 'MMP11',
 'MX1',
 'PITX1',
 'POSTN',
 'PRMT8',
 'PTBP1',
 'RPL26',
 'RPL32',
 'RPLP1',
 'RPS12',
 'RPS17',
 'RPS19',
 'RPS4X',
 'S100A8',
 'S100A9',
 'SUPT7L',
 'TMEM106C',
 'USP17L3',
 'XAF1']

In [93]:
SVGs_panck = sorted(SVGs_panck)
SVGs_panck

['ADAM15',
 'ALDOA',
 'ANXA8',
 'ATP5MC2',
 'C3',
 'CD44',
 'CHI3L1',
 'CRABP2',
 'CTHRC1',
 'CXCL14',
 'ELF5',
 'ENO1',
 'FBXO32',
 'FTH1',
 'HLA-A',
 'HLA-B',
 'HLA-C',
 'HLA-DPA1',
 'HLA-E',
 'HLA-F',
 'IFI27',
 'IFI6',
 'IFITM1',
 'LGALS1',
 'LTF',
 'MAZ',
 'MIF',
 'MMP7',
 'MUC1',
 'MYL12B',
 'MZT2B',
 'PRMT8',
 'PTBP1',
 'RPL12',
 'RPL13',
 'RPL18',
 'RPL35',
 'RPLP1',
 'RPS19',
 'RPS3A',
 'RPS5',
 'RPS6',
 'S100A7',
 'SCARA3',
 'SUPT7L',
 'TM4SF1',
 'TMEM106C',
 'TMSB4X',
 'TPI1',
 'TUBB4B']

Check whether any of the contamination genes are in the SVGs

In [94]:
# SVGs_cd45 = ["MGP","RPS17","PTBP1","FN1","POSTN","LAPTM4B","RPS19","RPS4X","RPL32","RPS12","HSPA8","XAF1","CLU","COTL1","COL3A1","LGALS1","AEBP1","BST2","GPNMB","IFI44","IFI44L","EIF4A2","BGN","FDCSP"]
# SVGs_panck = ["CRABP2","HLA-DPA1","MAZ","RPS19","TM4SF1","TUBB4B","LGALS1","MZT2B","ELF5","CTHRC1","IFI6","IFITM1","CD44","S100A7","TMSB4X","RPS5","SCARA3","RPS3A","RPL35","RPS6","MMP7","HLA-E","FBXO32","CHI3L1","C3","MYL12B"]
# SVGs_cd45 = sorted(SVGs_cd45)
# SVGs_panck = sorted(SVGs_panck)

In [95]:
set(SVGs_panck).intersection(set(contamination_genes))

set()

In [96]:
svgs_df_panck = log_transformed_expr.loc[rois_panck_train,SVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
svgs_df_cd45 = log_transformed_expr.loc[rois_cd45_train,SVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [97]:
#svgs_df_cd45[new_svg_cd45]

In [98]:

####svgs_df_panck = expression_matrix.T.loc[rois_panck,SVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
####svgs_df_cd45 = expression_matrix.T.loc[rois_cd45,SVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [99]:
svgs_df_panck.to_csv(os.path.join(outcome_path,f"svg_panck_raw.csv"), index=False)
svgs_df_cd45.to_csv(os.path.join(outcome_path,f"svg_cd45_raw.csv"), index=False)

TEST SET

In [100]:
svgs_df_panck = log_transformed_expr.loc[rois_panck_test,SVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
svgs_df_cd45 = log_transformed_expr.loc[rois_cd45_test,SVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})
svgs_df_panck.to_csv(os.path.join(outcome_path,f"svg_panck_raw_TEST.csv"), index=False)
svgs_df_cd45.to_csv(os.path.join(outcome_path,f"svg_cd45_raw_TEST.csv"), index=False)

Check overlap between SVGs and HVGs

In [101]:
set(SVGs_panck).intersection(set(HVGs_panck))

{'CHI3L1', 'LTF', 'MMP7', 'S100A7'}

In [102]:
set(SVGs_cd45).intersection(set(HVGs_cd45))

{'FDCSP', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGKC', 'MGP', 'S100A8'}

In [103]:
#old_hvgs = HVGs_cd45 + HVGs_panck

In [104]:
#old_svgs = SVGs_cd45 + SVGs_panck

In [105]:
# new_hvgs = HVGs_cd45 + HVGs_panck
# new_svgs = SVGs_cd45 + SVGs_panck
# len(set(old_hvgs).intersection(set(new_hvgs)))
#len(set(old_svgs).intersection(set(new_svgs)))

## Gene Panels

In [106]:
gene_panel_cd45 = ['CD274','CXCL9','IFNG','BCL2','CD3D', 'CD8A', 'CD4', 'FOXP3', 'CD68', 'CD163','IDO1', 'LAG3', 'STAT1', 'S100A9']
gene_panel_panck = ['BRCA1','BRCA2', 'TP53', 'PIK3CA','MYC','EGFR', 'AR', 'PTEN', 'RB1', 'CD274','BCL2','KRT5', 'KRT14', 'TP63', 'MKI67', 'VIM', 'MMP9','SOX10', 'ALDH1A1', 'CD44', 'ZEB1', 'SNAI1', 'CCNB1']

panel_df_panck = log_transformed_expr.loc[rois_panck_train,gene_panel_panck].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_cd45 = log_transformed_expr.loc[rois_cd45_train,gene_panel_cd45].reset_index().rename(columns={"index": "ROI_ID"})

####panel_df_panck = expression_matrix.T.loc[rois_panck,gene_panel_panck].reset_index().rename(columns={"index": "ROI_ID"})
####panel_df_cd45 = expression_matrix.T.loc[rois_cd45,gene_panel_cd45].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_panck.to_csv(os.path.join(outcome_path,f"panel_panck_raw.csv"), index=False)
panel_df_cd45.to_csv(os.path.join(outcome_path,f"panel_cd45_raw.csv"), index=False)

# TEST SET
panel_df_panck = log_transformed_expr.loc[rois_panck_test,gene_panel_panck].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_cd45 = log_transformed_expr.loc[rois_cd45_test,gene_panel_cd45].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_panck.to_csv(os.path.join(outcome_path,f"panel_panck_raw_TEST.csv"), index=False)
panel_df_cd45.to_csv(os.path.join(outcome_path,f"panel_cd45_raw_TEST.csv"), index=False)

# Check overlap of genes

In [107]:
table_path = os.path.join(root_path, "tables")
os.makedirs(table_path, exist_ok=True)

In [108]:
df_panel = pd.DataFrame(data = panel_df_panck.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_panel['Geneset'] = 'Panel'
df_hvg = pd.DataFrame(data = hvgs_df_panck.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_hvg['Geneset'] = 'HVG'
df_svg = pd.DataFrame(data = svgs_df_panck.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_svg['Geneset'] = 'SVG'
df_panck = pd.concat([df_panel,df_hvg,df_svg]).reset_index(drop=True)
df_panck['Segment'] = 'PanCK+'
df_panck.shape

(123, 3)

In [109]:
duplicated_panck = list(df_panck[df_panck.duplicated('Gene')].Gene.unique())
print(f"{len(duplicated_panck)} genes are duplicated in panck")
print(duplicated_panck)
df_panck[df_panck['Gene'].isin(duplicated_panck)].sort_values(['Gene'])

7 genes are duplicated in panck
['KRT14', 'KRT5', 'CD44', 'CHI3L1', 'LTF', 'MMP7', 'S100A7']


,Gene,Geneset,Segment
19,CD44,Panel,PanCK+
78,CD44,SVG,PanCK+
56,CHI3L1,HVG,PanCK+
79,CHI3L1,SVG,PanCK+
12,KRT14,Panel,PanCK+
25,KRT14,HVG,PanCK+
11,KRT5,Panel,PanCK+
63,KRT5,HVG,PanCK+
65,LTF,HVG,PanCK+
97,LTF,SVG,PanCK+


In [110]:
unique_panck = list(df_panck['Gene'].unique())
print(f"Total genes (PanCK): {df_panck.shape[0]}")
print(f"Duplicate genes (PanCK): {len(duplicated_panck)} ")
print(f"Unique genes (PanCK): {len(unique_panck)}")

Total genes (PanCK): 123
Duplicate genes (PanCK): 7 
Unique genes (PanCK): 116


In [111]:
df_panel = pd.DataFrame(data = panel_df_cd45.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_panel['Geneset'] = 'Panel'
df_hvg = pd.DataFrame(data = hvgs_df_cd45.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_hvg['Geneset'] = 'HVG'
df_svg = pd.DataFrame(data = svgs_df_cd45.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_svg['Geneset'] = 'SVG'
df_cd45 = pd.concat([df_panel,df_hvg,df_svg]).reset_index(drop=True)
df_cd45['Segment'] = 'CD45+'
df_cd45.shape

(114, 3)

In [112]:
duplicated_cd45 = list(df_cd45[df_cd45.duplicated('Gene')].Gene.unique())
print(f"{len(duplicated_cd45)} genes are duplicated in cd45")
print(duplicated_cd45)
df_cd45[df_cd45['Gene'].isin(duplicated_cd45)].sort_values(['Gene'])

9 genes are duplicated in cd45
['FDCSP', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGKC', 'MGP', 'S100A8', 'S100A9']


,Gene,Geneset,Segment
14,FDCSP,HVG,CD45+
76,FDCSP,SVG,CD45+
19,IGHG1,HVG,CD45+
84,IGHG1,SVG,CD45+
23,IGHG2,HVG,CD45+
85,IGHG2,SVG,CD45+
29,IGHG3,HVG,CD45+
86,IGHG3,SVG,CD45+
22,IGHG4,HVG,CD45+
87,IGHG4,SVG,CD45+


In [113]:
unique_cd45 = list(df_cd45['Gene'].unique())


df_cd45.shape

In [114]:

print(f"Total genes (CD45): {df_cd45.shape[0]}")
print(f"Duplicate genes (CD45): {len(duplicated_cd45)} ")
print(f"Unique genes (CD45): {len(unique_cd45)}")

Total genes (CD45): 114
Duplicate genes (CD45): 9 
Unique genes (CD45): 105


In [115]:
print(f"CD45+ # unique genes: {len(unique_cd45)}")
print(f"PanCK+ # unique genes: {len(unique_panck)}")


CD45+ # unique genes: 105
PanCK+ # unique genes: 116


In [116]:
print(f"# genes across both segments: {len(set(unique_panck).union(set(unique_cd45)))}")

# genes across both segments: 182


In [117]:
df_summary = pd.DataFrame({
    "Compartment": ["CD45", "PanCK"],
    "Total genes": [df_cd45.shape[0], df_panck.shape[0]],
    "Duplicated genes": [len(duplicated_cd45), len(duplicated_panck)],
    "Unique genes": [len(unique_cd45), len(unique_panck)],
    "Specific genes": [
        len(set(unique_cd45).difference(set(unique_panck))),
        len(set(unique_panck).difference(set(unique_cd45))),
    ],
    "inboth": [
        len(set(unique_panck).intersection(set(unique_cd45))),
        len(set(unique_panck).intersection(set(unique_cd45)))],
})

df_summary.loc[len(df_summary)] = [
    "Total",
    df_summary["Total genes"].sum(),
    df_summary["Duplicated genes"].sum(),
    df_summary["Unique genes"].sum(),
    df_summary["Specific genes"].sum(),
    df_summary["inboth"].sum(),
]

In [118]:
df_summary

,Compartment,Total genes,Duplicated genes,Unique genes,Specific genes,inboth
0,CD45,114,9,105,66,39
1,PanCK,123,7,116,77,39
2,Total,237,16,221,143,78


In [119]:
df_summary.to_csv(os.path.join(table_path,'ST1.csv'), index=False)

In [120]:
print(f"# genes in both segments: {len(set(unique_panck).intersection(set(unique_cd45)))}")

# genes in both segments: 39


In [121]:
print(f"# genes only in CD45+: {len(set(unique_cd45).difference(set(unique_panck)))}")
print(f"# genes only in PanCK+: {len(set(unique_panck).difference(set(unique_cd45)))}")


# genes only in CD45+: 66
# genes only in PanCK+: 77


In [122]:
print(f"genes only in CD45+: {(set(unique_cd45).difference(set(unique_panck)))}")
print(f"genes only in PanCK+: {(set(unique_panck).difference(set(unique_cd45)))}")


genes only in CD45+: {'AZGP1', 'IGKC', 'IGLL5', 'CLU', 'RPL32', 'EYA2', 'EIF4A2', 'CD4', 'CD8A', 'LAPTM4B', 'IGHG2', 'RPS4X', 'IDO1', 'RPS12', 'APOE', 'MMP11', 'SMR3A', 'AEBP1', 'RPL26', 'RPS17', 'ZFP57', 'TPSAB1', 'KRT15', 'CD3D', 'BGN', 'PPP2R2B', 'IGHG3', 'FOXP3', 'LAG3', 'JCHAIN', 'TRPM3', 'CD163', 'CXCL9', 'COL1A1', 'MMP1', 'COTL1', 'FXYD3', 'ISG15', 'KRT7', 'S100A1', 'COL3A1', 'CD68', 'C1QL2', 'IGHM', 'IFI44', 'HSPA8', 'CRYAB', 'RPL10L', 'XAF1', 'USP17L3', 'IGHG4', 'IGHG1', 'STAT1', 'IGHA1', 'POSTN', 'SPP1', 'SFRP1', 'AREG', 'BST2', 'PITX1', 'GPNMB', 'IFNG', 'FN1', 'MX1', 'CCL24', 'IFI44L'}
genes only in PanCK+: {'SOX10', 'STAC2', 'PTEN', 'RPL18', 'RARRES1', 'CTHRC1', 'IRX1', 'HLA-C', 'LTF', 'UBD', 'IRS4', 'ZEB1', 'BRCA1', 'BRCA2', 'HLA-F', 'HLA-DPA1', 'MAZ', 'SCGB2A2', 'RPS6', 'CHI3L1', 'ALDOA', 'MIF', 'RPL12', 'EGFR', 'S100A2', 'TM4SF1', 'CXCL1', 'CXCL14', 'AQP3', 'RPL35', 'C3', 'CCL28', 'FTH1', 'S100P', 'KLK5', 'AR', 'CYP1B1', 'TMSB4X', 'SNAI1', 'TP63', 'IFI27', 'RPS5', 'KRT16

# Save Standard Deviation
useful for calculating nRMSE

In [123]:
print(len(set(gene_panel_panck + SVGs_panck )))
print(len(gene_panel_panck) + len(SVGs_panck))
print(set(gene_panel_panck).intersection(set(SVGs_panck)))

72
73
{'CD44'}


In [124]:
allgenes_panck = list(set(gene_panel_panck + HVGs_panck + SVGs_panck))
allgenes_cd45 = list(set(gene_panel_cd45 + HVGs_cd45 + SVGs_cd45))

In [125]:
extra_cd45 = ['CALML5', 'CXCL14']
extra_panck = ['ANKRD30A', 'FOXA1', 'IGF2', 'MSLN', 'TFF3']
allgenes_panck = allgenes_panck + extra_panck
allgenes_cd45 = allgenes_cd45 + extra_cd45

In [126]:
std_panck_test = pd.DataFrame(log_transformed_expr.loc[rois_panck_test,allgenes_panck].std(axis=0),columns=['std'])
std_cd45_test = pd.DataFrame(log_transformed_expr.loc[rois_cd45_test,allgenes_cd45].std(axis=0),columns=['std'])
std_panck_test.index.name='Gene'
std_cd45_test.index.name='Gene'
std_panck_test.to_csv(os.path.join(outcome_path,f"allgenes_panck_test_std.csv"), index=True)
std_cd45_test.to_csv(os.path.join(outcome_path,f"allgenes_cd45_test_std.csv"), index=True)

# Additional

I want to know which leap ids have high variance on the genes of interest

CD45:
Panel ['CXCL9', 'CD68', 'STAT1', 'S100A9']
['TENT5C',
 'CCSAP',
 'ST6GALNAC3',
 'UBE2E1',
 'CD82',
 'DMRTC1B']
PanCK:
Panel ['PIK3CA', 'EGFR', 'MKI67']
['MKI67',
 'PIK3CA',
 'CA9',
 'LCN2',
 'AQP3',
 'IGHM']

 genes_of_interest[('CD45+','Panel')] = ['BCL2', 'IDO1', 'STAT1', 'CD274', 'CD3D', 'CXCL9'] #Panel
genes_of_interest[('CD45+','HVG')] = ['AREG','FDCSP','CPB1']
genes_of_interest[('CD45+','SVG')] = ['ST6GALNAC3','PRMT8','C7','TARP', 'TENT5C','RPL41','P2RY12','PITX1','CCSAP' ] #/SVG
genes_of_interest[('PanCK+','Panel')] = ['BRCA1', 'SNAI1', 'PIK3CA', 'BRCA2', 'EGFR', 'SNAI1','PIK3CA', 'BRCA2','EGFR'] #Panel
genes_of_interest[('PanCK+','HVG')] = ['IGHG1', 'IGHG4', 'IGHG3', 'IGKC',  'IGHG2',   'CSN3', 'CA9', 'CPB1' ]
genes_of_interest[('PanCK+','SVG')] = ['ATP5MC2', 'BRINP3', 'DMRTB1', 'NWD2', 'ENO1']

In [127]:
cd45_genes_of_interest = ['CXCL9', 'CD68', 'STAT1', 'S100A9','TENT5C','CCSAP', 'ST6GALNAC3', 'UBE2E1', 'CD82', 'DMRTC1B']
panck_genes_of_interest = ['PIK3CA', 'EGFR', 'MKI67', 'CA9', 'LCN2', 'AQP3', 'IGHM']

In [128]:
cd45_genes_of_interest = ['BCL2', 'IDO1', 'STAT1', 'CD274','AREG','FDCSP','CPB1','ST6GALNAC3','PRMT8','C7','TARP']
panck_genes_of_interest = ['BRCA1', 'SNAI1', 'PIK3CA', 'IGHG1', 'IGHG4', 'IGKC','ATP5MC2', 'BRINP3', 'DMRTB1','BRCA2', 'EGFR']

In [129]:
expr_of_interest_panck = log_transformed_expr.loc[rois_panck_test,panck_genes_of_interest]
expr_of_interest_cd45 = log_transformed_expr.loc[rois_cd45_test,cd45_genes_of_interest]

In [130]:
expr_of_interest_cd45 = sample_metadata.loc[rois_cd45_test,['LEAP_ID']].join(expr_of_interest_cd45,how='left')
expr_of_interest_panck = sample_metadata.loc[rois_panck_test,['LEAP_ID']].join(expr_of_interest_panck,how='left')


In [131]:
for gene in cd45_genes_of_interest:
    print(f"{gene}     \t{list(expr_of_interest_cd45.groupby('LEAP_ID').std().sort_values(gene, ascending=False).index[:5])}")

BCL2     	['LEAP120', 'LEAP079', 'LEAP121', 'LEAP078', 'LEAP094']
IDO1     	['LEAP120', 'LEAP079', 'LEAP121', 'LEAP082', 'LEAP078']
STAT1     	['LEAP121', 'LEAP120', 'LEAP079', 'LEAP078', 'LEAP021']
CD274     	['LEAP079', 'LEAP120', 'LEAP121', 'LEAP078', 'LEAP082']
AREG     	['LEAP120', 'LEAP121', 'LEAP079', 'LEAP078', 'LEAP094']
FDCSP     	['LEAP078', 'LEAP082', 'LEAP120', 'LEAP079', 'LEAP121']
CPB1     	['LEAP079', 'LEAP120', 'LEAP078', 'LEAP045', 'LEAP121']
ST6GALNAC3     	['LEAP079', 'LEAP120', 'LEAP078', 'LEAP094', 'LEAP093']
PRMT8     	['LEAP078', 'LEAP121', 'LEAP120', 'LEAP079', 'LEAP094']
C7     	['LEAP120', 'LEAP079', 'LEAP121', 'LEAP078', 'LEAP021']
TARP     	['LEAP079', 'LEAP120', 'LEAP078', 'LEAP121', 'LEAP094']


In [132]:
for gene in panck_genes_of_interest:
    print(f"{gene}     \t{list(expr_of_interest_panck.groupby('LEAP_ID').std().sort_values(gene, ascending=False).index[:5])}")

BRCA1     	['LEAP065', 'LEAP121', 'LEAP082', 'LEAP120', 'LEAP093']
SNAI1     	['LEAP065', 'LEAP121', 'LEAP120', 'LEAP093', 'LEAP082']
PIK3CA     	['LEAP065', 'LEAP082', 'LEAP094', 'LEAP121', 'LEAP120']
IGHG1     	['LEAP094', 'LEAP082', 'LEAP121', 'LEAP065', 'LEAP064']
IGHG4     	['LEAP094', 'LEAP082', 'LEAP121', 'LEAP065', 'LEAP064']
IGKC     	['LEAP094', 'LEAP082', 'LEAP065', 'LEAP064', 'LEAP120']
ATP5MC2     	['LEAP065', 'LEAP082', 'LEAP120', 'LEAP064', 'LEAP094']
BRINP3     	['LEAP065', 'LEAP094', 'LEAP082', 'LEAP120', 'LEAP064']
DMRTB1     	['LEAP065', 'LEAP121', 'LEAP120', 'LEAP079', 'LEAP094']
BRCA2     	['LEAP065', 'LEAP121', 'LEAP094', 'LEAP120', 'LEAP082']
EGFR     	['LEAP120', 'LEAP065', 'LEAP121', 'LEAP094', 'LEAP082']


best for CD45
LEAP120
LEAP079
LEAP121

best for PanCK
LEAP121
LEAP065
LEAP082
LEAP120

In [133]:
log_transformed_expr.loc[rois_cd45_test,allgenes_cd45]

,AZGP1,ADAM15,RPL32,FDCSP,EYA2,SMR3A,RPS17,CRABP1,ZFP57,TPSAB1,...,GABRP,RPS4X,IGHM,PTBP1,IGHG1,SFRP1,GPNMB,IFNG,CALML5,CXCL14
DSP-1001660013512-B-A06.dcc,0.693147,4.672829,2.564949,1.098612,1.386294,1.098612,2.772589,0.693147,0.693147,2.397895,...,1.791759,2.772589,0.693147,6.792344,1.098612,0.693147,1.386294,0.693147,1.098612,0.693147
DSP-1001660013512-B-A10.dcc,0.693147,5.669881,3.737670,1.791759,1.098612,1.386294,4.174387,0.693147,1.098612,1.098612,...,0.693147,3.526361,1.386294,7.584265,0.693147,1.791759,1.945910,0.693147,1.098612,2.197225
DSP-1001660013512-B-A12.dcc,0.693147,4.934474,3.091042,1.945910,0.693147,1.098612,3.761200,0.693147,1.098612,1.945910,...,0.693147,3.688879,0.693147,7.233455,0.693147,1.098612,1.609438,1.098612,0.693147,1.386294
DSP-1001660013512-B-B06.dcc,0.693147,5.398163,2.079442,1.098612,0.693147,1.386294,3.044522,1.386294,1.609438,1.609438,...,0.693147,2.639057,4.007333,6.871091,6.263398,1.609438,1.098612,1.098612,1.386294,0.693147
DSP-1001660013512-B-B08.dcc,0.693147,5.556828,3.091042,0.693147,1.098612,1.098612,3.091042,0.693147,1.098612,1.945910,...,1.098612,3.178054,4.477337,7.138867,6.955593,0.693147,1.386294,1.098612,0.693147,0.693147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DSP-1001660020815-G-H03.dcc,0.693147,5.866468,3.828641,1.098612,1.098612,0.693147,4.174387,1.098612,4.382027,2.484907,...,1.386294,3.806662,5.446737,7.584773,8.096513,1.609438,2.564949,1.098612,1.945910,1.609438
DSP-1001660020815-G-H05.dcc,1.386294,6.561031,4.828314,2.197225,1.945910,1.386294,5.056246,2.302585,5.062595,2.197225,...,2.484907,4.488636,4.890349,8.292799,9.000483,3.135494,2.833213,2.397895,2.772589,2.079442
DSP-1001660020815-G-H07.dcc,0.693147,5.030438,2.890372,0.693147,0.693147,0.693147,3.367296,1.098612,3.583519,0.693147,...,0.693147,2.484907,3.737670,6.675823,7.248504,0.693147,0.693147,0.693147,1.386294,0.693147
DSP-1001660020815-G-H09.dcc,0.693147,7.301822,4.605170,1.609438,2.079442,2.197225,4.976734,2.197225,5.655992,2.302585,...,2.079442,4.653960,5.568345,8.902320,9.587063,2.484907,2.772589,2.079442,2.564949,1.945910


# END
